In [0]:
dbutils.widgets.text('p_batch_id', '')
v_batch_id = dbutils.widgets.get('p_batch_id')

In [0]:
%run ../common/environmental_config

In [0]:
%run ../common/silver-helper

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.results"
silver_table = f"{catalog_name}.{silver_schema}.results"

In [0]:
from pyspark.sql import functions as F

In [0]:
df_results = (
    (spark.read.table(bronze_table).filter(F.col("batch_id") == v_batch_id))
    .drop('url')
    .withColumnRenamed('constructorId', 'constructor_id')
    .withColumnRenamed('driverId', 'driver_id')
    .withColumnRenamed('raceName', 'race_name')
    .withColumnRenamed('positionText', 'finish_position_text')
    .withColumnRenamed('date', 'race_date')
    .withColumnRenamed('grid', 'grid_position')
    .withColumnRenamed('laps', 'completed_laps')
    .withColumnRenamed('number', 'car_number')
    .withColumnRenamed('position', 'finish_position')
    .filter(
        (F.col('season').isNotNull()) &
        (F.col('round').isNotNull()) &
        (F.col('constructor_id').isNotNull()) &
        F.col('driver_id').isNotNull()
        )
    .dropDuplicates(['season' , 'round' , 'constructor_id' , 'driver_id'])
    .withColumn('race_name' , F.initcap(F.col('race_name')))
)


In [0]:
write_to_silver(input_df=df_results,
                target_table=silver_table,
                merge_condition="t.season = s.season AND t.round = s.round AND t.constructor_id = s.constructor_id AND t.driver_id = s.driver_id",
                columns_to_update=['race_date',
                                   'race_name',
                                   'grid_position',
                                   'completed_laps',
                                   'car_number',
                                   'finish_position',
                                   'points',
                                   'finish_position_text',
                                   'status',
                                   'ingestion_timestamp',
                                   'source_file',
                                   'batch_id']
)
